# Potencia de una central eléctrica según la temperatura
**Regresión lineal simple · nivel básico** · Proyecto 1 de *Mente Máquina*

Vamos a predecir la **potencia eléctrica** (MW) que genera una central a partir de **un solo dato: la temperatura ambiente**.
Usamos datos reales de una central de ciclo combinado (UCI, conjunto nº 294, licencia CC BY 4.0).

**Cómo usar este cuaderno:** ejecuta las celdas de arriba abajo, una a una, con el botón ▶ (o `Mayús + Intro`).
Lee antes de cada celda de código qué hace y por qué. No hace falta saber programar: solo ejecutar y mirar el resultado.

## Preparación: instalar las bibliotecas
Una biblioteca es un conjunto de herramientas ya hechas. Aquí instalamos `ucimlrepo` (para descargar el conjunto de datos) y cargamos las demás.

In [ ]:
!pip install -q ucimlrepo

import pandas as pd                       # pandas: manejar tablas de datos
import numpy as np                        # numpy: cálculo con números
import matplotlib.pyplot as plt           # matplotlib: dibujar gráficos
from ucimlrepo import fetch_ucirepo      # para descargar el conjunto de datos de UCI
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm              # statsmodels: estadística de la regresión

print("Todo listo")

## Paso 2 · Descargar y abrir los datos
**Qué hacemos:** descargar la tabla de la central y quedarnos con dos columnas.
**Cómo:** `fetch_ucirepo(id=294)` pide el conjunto nº 294 de UCI. Nos quedamos con la temperatura ambiente (`AT`) y la potencia neta (`PE`).
**Por qué:** una regresión simple usa una entrada (x) y una salida (y). Aquí x = temperatura e y = potencia.
**Para qué:** tener los datos en una tabla (`df`) con la que trabajar en todos los pasos siguientes.

In [ ]:
ccpp = fetch_ucirepo(id=294)              # descargar el conjunto de datos nº 294

entradas = ccpp.data.features              # columnas de entrada (AT, V, AP, RH)
salida = ccpp.data.targets                 # columna de salida (PE)

# Nos quedamos solo con la temperatura (x) y la potencia (y), con nombres claros
df = pd.DataFrame({
    "Temperatura": entradas["AT"],         # °C
    "Potencia": salida["PE"],              # MW
})

df = df.dropna()                           # quitar filas incompletas, si las hubiera

print("Filas y columnas:", df.shape)

## Paso 3 · Revisar la tabla
**Qué hacemos:** mirar las primeras filas y un resumen de números.
**Cómo:** `head()` enseña las 5 primeras filas; `describe()` calcula mínimo, máximo, media…
**Por qué:** antes de calcular nada hay que *conocer* los datos: sus unidades, sus rangos y si hay valores raros.
**Para qué:** saber entre qué temperaturas es válido nuestro modelo (esto lo usaremos en el paso 9).

In [ ]:
print(df.head())                            # las 5 primeras filas
print()
print(df.describe().round(2))              # resumen: media, mínimo, máximo...
print()
print("Valores que faltan:")
print(df.isna().sum())

## Paso 4 · Dibujar el diagrama de dispersión
**Qué hacemos:** dibujar un punto por cada registro: temperatura en horizontal, potencia en vertical.
**Cómo:** `plt.scatter(x, y)`.
**Por qué:** hay que *mirar antes de calcular*. Si los puntos forman una línea, una recta es buena idea; si forman una curva, no.
**Para qué:** ver la tendencia a ojo. Esperamos una relación **negativa**: más calor, menos potencia.

In [ ]:
plt.figure(figsize=(9, 5.5))
plt.scatter(df["Temperatura"], df["Potencia"], alpha=0.3)   # alpha: transparencia, para ver dónde se acumulan
plt.xlabel("Temperatura ambiente (°C)")
plt.ylabel("Potencia eléctrica (MW)")
plt.title("Temperatura frente a potencia")
plt.grid(True)
plt.show()

# Un número que resume la relación: el coeficiente de correlación (de -1 a +1)
print("Correlación:", round(df["Temperatura"].corr(df["Potencia"]), 3))

## Paso 5 · Separar entrenamiento y prueba
**Qué hacemos:** dividir los datos en dos grupos: 80 % para entrenar y 20 % para examinar.
**Cómo:** `train_test_split(..., test_size=0.20, random_state=42)`.
**Por qué:** si evaluamos el modelo con los mismos datos con los que aprendió, es como ponerle un examen con las preguntas ya vistas. Los datos de prueba son «preguntas nuevas».
**Para qué:** medir con honestidad cuánto se equivoca. `random_state=42` hace que el reparto sea siempre el mismo.

In [ ]:
X = df[["Temperatura"]]                    # entrada (tabla de una columna)
y = df["Potencia"]                         # salida

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,                        # 20 % para prueba
    random_state=42,                       # repartir siempre igual
)

print("Datos de entrenamiento:", len(X_train))
print("Datos de prueba:", len(X_test))

## Paso 6 · Ajustar la recta
**Qué hacemos:** pedirle al ordenador que encuentre la mejor recta `potencia = m · temperatura + b`.
**Cómo:** `LinearRegression().fit(...)` aplica el método de mínimos cuadrados que vimos en el paso 1.
**Por qué:** de todas las rectas posibles, la mejor es la que deja el menor error total (SSE).
**Para qué:** obtener los dos números del modelo: la pendiente **m** y el intercepto **b**.

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)               # «entrenar»: encontrar la mejor recta con los datos de entrenamiento

pendiente = modelo.coef_[0]                # m: MW que cambian por cada °C
intercepto = modelo.intercept_             # b: potencia estimada a 0 °C

print(f"Pendiente (m):   {pendiente:.4f} MW por °C")
print(f"Intercepto (b):  {intercepto:.4f} MW")
print(f"Ecuación:  Potencia = {pendiente:.4f} · Temperatura + {intercepto:.4f}")

## Paso 7 · Evaluar con datos nuevos
**Qué hacemos:** usar la recta para predecir los datos de prueba y medir el error.
**Cómo:** `predict(X_test)` y tres medidas: **R²**, **RMSE** y **MAE**.
**Por qué:** R² dice cuánto de la variación explica la recta; MAE es el error medio en MW; RMSE es parecido pero castiga más los errores grandes.
**Para qué:** saber si el modelo es fiable. Cuidado: R² **no** es un «porcentaje de aciertos».

In [ ]:
y_pred = modelo.predict(X_test)            # lo que predice la recta para los datos de prueba

comparacion = pd.DataFrame({
    "Temperatura": X_test["Temperatura"].values,
    "Potencia real": y_test.values,
    "Potencia predicha": y_pred.round(2),
})
print(comparacion.head(8))
print()

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("RESULTADOS DEL MODELO (datos de prueba)")
print(f"R²:    {r2:.4f}")
print(f"RMSE:  {rmse:.3f} MW")
print(f"MAE:   {mae:.3f} MW")

## Paso 8 · Mirar los errores y poner límites
**Qué hacemos:** dibujar la recta sobre los datos y representar los **residuos** (real − predicho).
**Cómo:** `plt.plot` para la recta; `plt.scatter(predicho, residuo)` para los errores.
**Por qué:** un buen R² no basta. Si los errores forman un patrón (una curva, un abanico), la recta se está dejando algo sin explicar.
**Para qué:** conocer las limitaciones del modelo antes de fiarnos de él.

In [ ]:
# 1) La recta sobre los datos de prueba
x_linea = np.linspace(X["Temperatura"].min(), X["Temperatura"].max(), 100).reshape(-1, 1)
x_linea = pd.DataFrame(x_linea, columns=["Temperatura"])
y_linea = modelo.predict(x_linea)

plt.figure(figsize=(9, 5.5))
plt.scatter(X_test["Temperatura"], y_test, alpha=0.35, label="Datos reales")
plt.plot(x_linea["Temperatura"], y_linea, color="red", linewidth=2, label="Recta de regresión")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Potencia (MW)")
plt.title("Modelo de regresión lineal simple")
plt.legend()
plt.grid(True)
plt.show()

# 2) Los residuos: lo que se le escapa a la recta
residuos = y_test - y_pred
plt.figure(figsize=(9, 4.5))
plt.scatter(y_pred, residuos, alpha=0.35)
plt.axhline(0, color="red", linestyle="--")       # línea de error cero
plt.xlabel("Potencia predicha (MW)")
plt.ylabel("Residuo (MW)")
plt.title("Análisis de residuos")
plt.grid(True)
plt.show()

## Paso 9 · Poner un rango a la predicción
**Qué hacemos:** predecir la potencia para 25 °C y dar un **intervalo**: «entre tanto y tanto, con un 95 % de confianza».
**Cómo:** con `statsmodels`, que además de la recta calcula la incertidumbre.
**Por qué:** una predicción sin margen de error parece más segura de lo que es. Y solo debemos predecir **dentro del rango** de temperaturas que vimos (paso 3).
**Para qué:** dar una respuesta honesta.

In [ ]:
# statsmodels necesita añadir explícitamente la columna del intercepto («const»)
X_stats = sm.add_constant(X_train)
modelo_stats = sm.OLS(y_train, X_stats).fit()

nueva = pd.DataFrame({"const": [1.0], "Temperatura": [25.0]})       # queremos predecir a 25 °C
pred = modelo_stats.get_prediction(nueva).summary_frame(alpha=0.05)  # 95 % de confianza

print(f"Temperatura: 25 °C")
print(f"Potencia estimada: {pred['mean'].iloc[0]:.1f} MW")
print(f"Rango del 95 %:    de {pred['obs_ci_lower'].iloc[0]:.1f} a {pred['obs_ci_upper'].iloc[0]:.1f} MW")
print()
print(modelo_stats.summary().tables[1])         # coeficientes con su intervalo de confianza del 95 %
print()
print(f"Rango de temperaturas con datos: {df['Temperatura'].min():.1f} a {df['Temperatura'].max():.1f} °C")
print("¿25 °C está dentro del rango?", df['Temperatura'].min() <= 25 <= df['Temperatura'].max())

## Paso 10 · Contar el resultado en cinco frases
**Qué hacemos:** resumir todo el proyecto con una plantilla de 5 frases: pregunta, datos, resultado, limitaciones y conclusión.
**Cómo:** el cuaderno rellena la plantilla con tus propios números.
**Por qué:** saber contar un resultado a otra persona es tan importante como calcularlo.
**Para qué:** copiar este texto en tu informe, en un correo o en la web.

In [ ]:
rango_ini, rango_fin = df["Temperatura"].min(), df["Temperatura"].max()
signo = "baja" if pendiente < 0 else "sube"

print("1. Pregunta: ¿cuánta potencia produce la central según la temperatura ambiente?")
print(f"2. Datos: {len(df)} registros reales de una central de ciclo combinado, con temperaturas entre {rango_ini:.1f} y {rango_fin:.1f} °C.")
print(f"3. Resultado: por cada grado más de temperatura, la potencia {signo} unos {abs(pendiente):.2f} MW (R² = {r2:.2f}: la temperatura explica el {r2*100:.0f} % de la variación).")
print(f"4. Limitaciones: solo es fiable entre {rango_ini:.0f} y {rango_fin:.0f} °C; con una sola variable y una sola central no explica todo (presión, humedad…).")
print(f"5. Conclusión: a 25 °C estimamos {pred['mean'].iloc[0]:.0f} MW, con un error típico de unos {mae:.1f} MW (MAE).")

## Extra · Preparar los datos para la web
Esta celda guarda los datos reales y los resultados en un archivo (`datos--potencia-central-regresion-simple.json`) y lo descarga.
Con él, la web dibuja la nube de puntos **real** y la recta que tú has calculado, y deja moverla con deslizadores.

In [ ]:
import json

datos_web = {
    "t": df["Temperatura"].round(2).tolist(),                 # temperaturas
    "p": df["Potencia"].round(2).tolist(),                    # potencias
    "e": df.index.isin(X_test.index).astype(int).tolist(),    # 1 = dato de prueba, 0 = entrenamiento
    "m": float(pendiente), "b": float(intercepto),            # la recta calculada por el programa
    "r2": float(r2), "rmse": float(rmse), "mae": float(mae),
}

nombre = "datos--potencia-central-regresion-simple.json"
with open(nombre, "w") as fichero:
    json.dump(datos_web, fichero)

from google.colab import files      # solo funciona en Google Colab
files.download(nombre)              # descarga el archivo a tu ordenador

## Retos para consolidar
1. **Básico:** cambia 25 °C por 15, 20, 30 y 35 °C en el paso 9 y compara las predicciones.
2. **Intermedio:** cambia `test_size` en el paso 5 (0.30, 0.10) y observa cómo cambian R², RMSE y MAE.
3. **Avanzado:** añade presión, humedad y vacío como entradas (`entradas[["AT","V","AP","RH"]]`) y compara con la regresión simple. Eso es la **regresión múltiple**, el siguiente proyecto.

> La regresión simple no demuestra que la temperatura sea la *única* causa de los cambios de potencia: la central depende de más variables.

*Datos: Combined Cycle Power Plant, UCI Machine Learning Repository (CC BY 4.0). Cuaderno de Mente Máquina · Pedro José de la Rosa Martín.*